# Kalshi Bet Probability Analysis -- Rotten Tomatoes Tomatometer

In [ ]:
import os
import warnings
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sqlalchemy import create_engine, text

warnings.filterwarnings("ignore", category=FutureWarning)

# ── Parameters ──────────────────────────────────────────────────────────────
MOVIE_SLUG = "project_hail_mary"
THRESHOLD = 0.90          # Kalshi bet threshold (e.g. 90%)
DIRECTION = "below"       # "below" = prob score drops below threshold; "above" = prob it rises above
HOURS_UNTIL_CLOSE = 48    # Forecast window in hours
ALPHA = 0.05              # Significance level for Poisson tail truncation
KDE_BANDWIDTH = "silverman"

# Optional manual overrides (set to None to use DB values)
OVERRIDE_CURRENT_SCORE = None   # e.g. 0.92
OVERRIDE_TOTAL_REVIEWS = None   # e.g. 150

## Step 1: Load Review Data

In [ ]:
# ── Database connection ────────────────────────────────────────────────────
DATABASE_URL = os.environ["DATABASE_URL"]
if DATABASE_URL.startswith("postgres://"):
    DATABASE_URL = DATABASE_URL.replace("postgres://", "postgresql://", 1)

engine = create_engine(DATABASE_URL)

query = text("""
    SELECT reviewer_name, tomatometer_sentiment, estimated_timestamp,
           scrape_time, timestamp_confidence, page_position, top_critic
    FROM reviews
    WHERE movie_slug = :slug
      AND estimated_timestamp IS NOT NULL
    ORDER BY estimated_timestamp ASC
""")

with engine.connect() as conn:
    df = pd.read_sql(query, conn, params={"slug": MOVIE_SLUG})

df["estimated_timestamp"] = pd.to_datetime(df["estimated_timestamp"], utc=True)
df["is_fresh"] = df["tomatometer_sentiment"] == "positive"

print(f"Loaded {len(df)} reviews for '{MOVIE_SLUG}'")

In [ ]:
# ── Current state ──────────────────────────────────────────────────────────
total_reviews = OVERRIDE_TOTAL_REVIEWS if OVERRIDE_TOTAL_REVIEWS is not None else len(df)
fresh_count = int(df["is_fresh"].sum())

if OVERRIDE_CURRENT_SCORE is not None and OVERRIDE_TOTAL_REVIEWS is not None:
    fresh_count = int(round(OVERRIDE_CURRENT_SCORE * OVERRIDE_TOTAL_REVIEWS))

current_score = fresh_count / total_reviews if total_reviews > 0 else 0.0

date_min = df["estimated_timestamp"].min()
date_max = df["estimated_timestamp"].max()
span_days = (date_max - date_min).total_seconds() / 86400
reviews_per_day = len(df) / span_days if span_days > 0 else 0

print(f"Total reviews : {total_reviews}")
print(f"Fresh count   : {fresh_count}")
print(f"Current score : {current_score:.1%}")
print(f"Date range    : {date_min:%Y-%m-%d %H:%M} → {date_max:%Y-%m-%d %H:%M}  ({span_days:.1f} days)")
print(f"Reviews / day : {reviews_per_day:.1f}")

## Step 2: Nonhomogeneous Poisson Process

In [ ]:
# ── Fit KDE on review arrival times ────────────────────────────────────────
t_origin = df["estimated_timestamp"].min()
arrival_hours = (df["estimated_timestamp"] - t_origin).dt.total_seconds() / 3600
arrival_hours = arrival_hours.values.astype(float)

now = datetime.now(timezone.utc)
t_now = (now - t_origin).total_seconds() / 3600
t_end = t_now + HOURS_UNTIL_CLOSE

kde = stats.gaussian_kde(arrival_hours, bw_method=KDE_BANDWIDTH)

# Integrate lambda(t) = N * kde(t) over [t_now, t_end]
N_obs = len(df)
t_grid = np.linspace(t_now, t_end, 500)
lambda_grid = N_obs * kde(t_grid)
Lambda = np.trapezoid(lambda_grid, t_grid)

# Simple fallback: count reviews in most recent HOURS_UNTIL_CLOSE window
recent_cutoff = now - pd.Timedelta(hours=HOURS_UNTIL_CLOSE)
Lambda_simple = (df["estimated_timestamp"] >= recent_cutoff).sum()

print(f"Lambda (KDE)    : {Lambda:.2f} expected reviews in next {HOURS_UNTIL_CLOSE}h")
print(f"Lambda (simple) : {Lambda_simple} reviews in most recent {HOURS_UNTIL_CLOSE}h")

if Lambda < 1:
    warnings.warn(f"Lambda = {Lambda:.2f} is very low — few reviews expected. Results may be unreliable.")

In [ ]:
# ── Visualize lambda(t) ────────────────────────────────────────────────────
t_full = np.linspace(0, t_end + 12, 1000)
lambda_full = N_obs * kde(t_full)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t_full, lambda_full, color="steelblue", linewidth=1.5, label="λ(t) = N · kde(t)")
ax.axvline(t_now, color="red", linestyle="--", alpha=0.7, label="Now")
ax.fill_between(t_grid, lambda_grid, alpha=0.25, color="orange",
                label=f"Forecast window (Λ = {Lambda:.2f})")

# Rug plot of actual arrivals
ax.plot(arrival_hours, np.zeros_like(arrival_hours), "|", color="grey", alpha=0.3, markersize=6)

ax.set_xlabel("Hours since first review")
ax.set_ylabel("Rate (reviews / hour)")
ax.set_title(f"Review arrival rate — {MOVIE_SLUG}")
ax.legend()
plt.tight_layout()
plt.show()

## Step 3: N-ceiling

In [ ]:
# ── Compute n-ceiling ──────────────────────────────────────────────────────
poisson_dist = stats.poisson(mu=Lambda)
n_ceiling = int(poisson_dist.ppf(1 - ALPHA)) + 1

print(f"Lambda          : {Lambda:.2f}")
print(f"n_ceiling       : {n_ceiling}")
print(f"P(N >= {n_ceiling})     : {1 - poisson_dist.cdf(n_ceiling - 1):.6f}")
print(f"Alpha           : {ALPHA}")

In [ ]:
# ── Visualize Poisson PMF ──────────────────────────────────────────────────
n_range = np.arange(0, n_ceiling + 5)
pmf_values = poisson_dist.pmf(n_range)

colors = ["orange" if n >= n_ceiling else "steelblue" for n in n_range]

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(n_range, pmf_values, color=colors, edgecolor="white", linewidth=0.5)
ax.axvline(n_ceiling - 0.5, color="red", linestyle="--", alpha=0.7, label=f"n_ceiling = {n_ceiling}")
ax.set_xlabel("n (number of new reviews)")
ax.set_ylabel("P(N = n)")
ax.set_title(f"Poisson PMF — Λ = {Lambda:.2f}")
ax.legend()
plt.tight_layout()
plt.show()

## Step 4: Estimate Fresh Review Probability

In [ ]:
# ── Estimate p(fresh) ──────────────────────────────────────────────────────
recent_window = now - pd.Timedelta(days=7)
recent_reviews = df[df["estimated_timestamp"] >= recent_window]

if len(recent_reviews) >= 5:
    p_fresh = recent_reviews["is_fresh"].mean()
    p_source = f"recent 7-day window ({len(recent_reviews)} reviews)"
else:
    p_fresh = current_score
    p_source = f"overall score (only {len(recent_reviews)} recent reviews)"

print(f"p(fresh)  : {p_fresh:.4f}")
print(f"Source    : {p_source}")

## Step 5: Probability of Breaking Threshold

In [ ]:
# ── Core calculation ───────────────────────────────────────────────────────
def compute_break_probability(current_fresh, current_total, threshold, direction,
                              p_fresh, poisson_dist, n_ceiling):
    """Compute the probability that the score breaks the threshold."""
    basket_matrix = np.zeros((n_ceiling + 1, n_ceiling + 1))
    total_prob = 0.0

    for n in range(1, n_ceiling + 1):
        p_n = poisson_dist.pmf(n)
        if p_n < 1e-15:
            continue
        binom_dist = stats.binom(n, p_fresh)
        for i in range(0, n + 1):
            new_score = (current_fresh + i) / (current_total + n)
            if direction == "below":
                breaks = new_score < threshold
            else:
                breaks = new_score > threshold
            if breaks:
                p_scenario = p_n * binom_dist.pmf(i)
                basket_matrix[n][i] = p_scenario
                total_prob += p_scenario

    return total_prob, basket_matrix


prob, basket_matrix = compute_break_probability(
    fresh_count, total_reviews, THRESHOLD, DIRECTION,
    p_fresh, poisson_dist, n_ceiling
)

print(f"{'═' * 50}")
print(f"  P(score {'drops below' if DIRECTION == 'below' else 'rises above'} {THRESHOLD:.0%}) = {prob:.4%}")
print(f"{'═' * 50}")

In [ ]:
# ── Heatmap of (n, i) probability contributions ───────────────────────────
# Trim to the active region
active = basket_matrix[1:n_ceiling + 1, :]
max_i = max(1, np.max(np.where(active > 0)[1]) + 2) if active.any() else n_ceiling
active = active[:, :max_i]

fig, ax = plt.subplots(figsize=(min(14, max_i * 0.6 + 2), min(10, n_ceiling * 0.4 + 1)))
im = ax.imshow(active, aspect="auto", origin="lower", cmap="YlOrRd")
ax.set_xlabel("i (fresh reviews out of n)")
ax.set_ylabel("n (total new reviews)")
ax.set_yticks(range(active.shape[0]))
ax.set_yticklabels(range(1, n_ceiling + 1))
ax.set_title(f"Scenario probability contributions — P(break) = {prob:.4%}")
plt.colorbar(im, ax=ax, label="P(N=n) · P(i fresh | n)")
plt.tight_layout()
plt.show()

In [ ]:
# ── Threshold boundary table ──────────────────────────────────────────────
rows = []
for n in range(1, n_ceiling + 1):
    for i in range(0, n + 1):
        new_score = (fresh_count + i) / (total_reviews + n)
        if DIRECTION == "below":
            breaks = new_score < THRESHOLD
        else:
            breaks = new_score > THRESHOLD
        if breaks:
            critical_i = i
            break
    else:
        critical_i = None

    if DIRECTION == "above":
        # For "above", find the minimum i that breaks
        critical_i = None
        for i in range(0, n + 1):
            new_score = (fresh_count + i) / (total_reviews + n)
            if new_score > THRESHOLD:
                critical_i = i
                break

    p_n = poisson_dist.pmf(n)
    if critical_i is not None:
        boundary_score = (fresh_count + critical_i) / (total_reviews + n)
        rows.append({
            "n": n,
            "P(N=n)": f"{p_n:.6f}",
            "critical_i": critical_i,
            "score_at_boundary": f"{boundary_score:.4%}",
            "P(break|n)": f"{sum(basket_matrix[n]):.6f}",
        })
    else:
        rows.append({
            "n": n,
            "P(N=n)": f"{p_n:.6f}",
            "critical_i": "impossible",
            "score_at_boundary": "—",
            "P(break|n)": "0",
        })

boundary_df = pd.DataFrame(rows)
boundary_df

## Summary

In [ ]:
print(f"Movie           : {MOVIE_SLUG}")
print(f"Current score   : {current_score:.1%}  ({fresh_count}/{total_reviews})")
print(f"Threshold       : {THRESHOLD:.0%} ({DIRECTION})")
print(f"Forecast window : {HOURS_UNTIL_CLOSE}h")
print(f"p(fresh)        : {p_fresh:.4f}  ({p_source})")
print(f"Lambda (KDE)    : {Lambda:.2f}")
print(f"n_ceiling       : {n_ceiling}  (alpha = {ALPHA})")
print()
print(f"{'═' * 50}")
print(f"  P(score {'drops below' if DIRECTION == 'below' else 'rises above'} {THRESHOLD:.0%}) = {prob:.4%}")
print(f"{'═' * 50}")